In [28]:
# Welcome to your new notebook
# Type here in the cell editor to add code!


StatementMeta(, 8f453ac4-3c3a-42be-a14c-83dfa3de5248, 30, Finished, Available, Finished, False)

# silver_telemetry
1. fact_telemetry
2. fact_energy
3. dim_date
# silver_events
1. fact_event
# silver_asset_metadata
1. - dim_site
2. - dim_building
3. - dim_asset

In [29]:
display(
    spark.sql("""
        CREATE OR REPLACE TABLE dbo.gold_dim_site AS
        SELECT DISTINCT
            site_id
        FROM dbo.silver_asset_metadata
        WHERE site_id IS NOT NULL
          AND TRIM(site_id) <> ''
          Order by site_id ASC
    """)
)


StatementMeta(, 8f453ac4-3c3a-42be-a14c-83dfa3de5248, 31, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, f3b7452f-3c69-4c57-89a0-637fd9d2b7d9)

# we  build the building dimension from the telemetry data.

In [30]:
display(
    spark.sql("""
        CREATE OR REPLACE TABLE dbo.gold_dim_building AS

        SELECT DISTINCT
            building_id,
            site_id
            

        FROM dbo.silver_telemetry

        WHERE site_id IS NOT NULL
          AND building_id IS NOT NULL
          AND TRIM(site_id) <> ''
          AND TRIM(building_id) <> ''
          Order by building_id ASC
    """)
)

StatementMeta(, 8f453ac4-3c3a-42be-a14c-83dfa3de5248, 32, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, d92455f8-92dd-423a-8697-ff5bb1738c83)

# asset metadata is the best source for the asset dimension.

In [31]:
display(
    spark.sql("""
        CREATE OR REPLACE TABLE dbo.gold_dim_asset AS

        SELECT DISTINCT
            asset_id,
            asset_name,
            asset_type,
            manufacturer,
            installation_date,
            site_id

        FROM dbo.silver_asset_metadata

        WHERE asset_id IS NOT NULL
          AND TRIM(asset_id) <> ''
          Order by asset_id ASC
    """)
)

StatementMeta(, 8f453ac4-3c3a-42be-a14c-83dfa3de5248, 33, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 58e4bf25-e8bf-4c7d-9e92-42f96b438d78)

# We  derive it from telemetry timestamps.

In [32]:
display(
    spark.sql("""
        CREATE OR REPLACE TABLE dbo.gold_dim_time AS

        SELECT DISTINCT
            DATE(timestamp) AS date_key,
            YEAR(timestamp) AS year,
            MONTH(timestamp) AS month,
            DAY(timestamp) AS day,
            DAYOFWEEK(timestamp) AS day_of_week,
            WEEKOFYEAR(timestamp) AS week_of_year,
            DAYOFYEAR(timestamp) AS day_of_year

        FROM dbo.silver_telemetry

        WHERE timestamp IS NOT NULL
    Order by date_key ASC
    """)
)

StatementMeta(, 8f453ac4-3c3a-42be-a14c-83dfa3de5248, 34, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, b2d42184-fd65-448c-a66f-9311a2008f8c)

# This is the main IoT telemetry fact.

In [33]:
display(
    spark.sql("""
        CREATE OR REPLACE TABLE dbo.gold_fact_telemetry
        USING DELTA
        PARTITIONED BY (date_key)
        AS

        SELECT
            ROW_NUMBER() OVER (
                ORDER BY
                    timestamp,
                    site_id,
                    building_id,
                    asset_id,
                    sensor_id
            ) AS telemetry_key,

            DATE(timestamp) AS date_key,
            timestamp,
            site_id,
            building_id,
            asset_id,
            sensor_id,

            temperature,
            humidity,
            pressure,
            vibration,
            power_consumption,
            operating_mode,

            source_file,
            ingestion_timestamp

        FROM dbo.dq_telemetry

        WHERE validation_status = 'VALID'
          AND timestamp IS NOT NULL
    """)
)

StatementMeta(, 8f453ac4-3c3a-42be-a14c-83dfa3de5248, 35, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, a60322b7-cc80-4ce0-be26-3533283177a5)

In [34]:
display(
    spark.sql("""
        CREATE OR REPLACE TABLE dbo.gold_fact_energy
        USING DELTA
        PARTITIONED BY (date_key)
        AS

        SELECT
            ROW_NUMBER() OVER (
                ORDER BY
                    DATE_TRUNC('hour', timestamp),
                    site_id,
                    building_id,
                    asset_id
            ) AS energy_key,

            DATE(timestamp) AS date_key,

            DATE_TRUNC('hour', timestamp) AS hour,

            site_id,
            building_id,
            asset_id,

            ROUND(
                SUM(power_consumption),
                2
            ) AS hourly_energy_consumption,

            ROUND(
                AVG(power_consumption),
                2
            ) AS avg_power_consumption,

            COUNT(*) AS reading_count

        FROM dbo.dq_telemetry

        WHERE validation_status = 'VALID'
          AND timestamp IS NOT NULL
          AND power_consumption IS NOT NULL

        GROUP BY
            DATE_TRUNC('hour', timestamp),
            DATE(timestamp),
            site_id,
            building_id,
            asset_id
    """)
)

StatementMeta(, 8f453ac4-3c3a-42be-a14c-83dfa3de5248, 36, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, c5331d27-492f-44d7-9d6c-79c78915bd84)

# events become an event fact.

In [35]:
display(
    spark.sql("""
        CREATE OR REPLACE TABLE dbo.gold_fact_event
        USING DELTA
        PARTITIONED BY (date_key)
        AS

        SELECT
            ROW_NUMBER() OVER (
                ORDER BY
                    timestamp,
                    event_id,
                    asset_id
            ) AS event_key,

            DATE(timestamp) AS date_key,
            timestamp,
            event_id,
            asset_id,
            event_type,
            severity,
            message,

            source_file,
            ingestion_timestamp

        FROM dbo.dq_events

        WHERE validation_status = 'VALID'
          AND timestamp IS NOT NULL
    """)
)

StatementMeta(, 8f453ac4-3c3a-42be-a14c-83dfa3de5248, 37, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, c2020962-c4bd-469e-84c7-6daa33abe0b9)

In [36]:
display(
    spark.sql("""
        SELECT 'gold_dim_asset' AS table_name, COUNT(*) AS record_count
        FROM dbo.gold_dim_asset

        UNION ALL

        SELECT 'gold_dim_building', COUNT(*)
        FROM dbo.gold_dim_building

        UNION ALL

        SELECT 'gold_dim_site', COUNT(*)
        FROM dbo.gold_dim_site

        UNION ALL

        SELECT 'gold_dim_time', COUNT(*)
        FROM dbo.gold_dim_time

        UNION ALL

        SELECT 'gold_fact_energy', COUNT(*)
        FROM dbo.gold_fact_energy

        UNION ALL

        SELECT 'gold_fact_event', COUNT(*)
        FROM dbo.gold_fact_event

        UNION ALL

        SELECT 'gold_fact_telemetry', COUNT(*)
        FROM dbo.gold_fact_telemetry
    """)
)

StatementMeta(, 8f453ac4-3c3a-42be-a14c-83dfa3de5248, 38, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 252954e9-1c49-4746-8e6d-bfa6061a6ae8)

In [37]:
display(spark.sql("SELECT * FROM dbo.gold_dim_asset"))
display(spark.sql("SELECT * FROM dbo.gold_dim_building"))
display(spark.sql("SELECT * FROM dbo.gold_dim_site"))
display(spark.sql("SELECT * FROM dbo.gold_dim_time"))
display(spark.sql("SELECT * FROM dbo.gold_fact_energy"))
display(spark.sql("SELECT * FROM dbo.gold_fact_event"))
display(spark.sql("SELECT * FROM dbo.gold_fact_telemetry"))

StatementMeta(, 8f453ac4-3c3a-42be-a14c-83dfa3de5248, 39, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, eee28aee-de5c-44dc-8ada-ba7589a60d55)

SynapseWidget(Synapse.DataFrame, 42f77c0d-bf7a-4393-bd05-cbb3ab6495cc)

SynapseWidget(Synapse.DataFrame, 3a253bf4-6a52-47f4-a127-18b5a276c982)

SynapseWidget(Synapse.DataFrame, b974294f-e7e7-45ca-a90e-38a93d4dcd0f)

SynapseWidget(Synapse.DataFrame, c73af911-1e07-4151-a78b-3ac757ef4b88)

SynapseWidget(Synapse.DataFrame, b570097a-2084-47f1-8543-be4af3baac7c)

SynapseWidget(Synapse.DataFrame, afa4a9e0-cc7c-4b3d-b95a-8558314acf58)

# ------------------------------------------------------

                   
# Dashboarding	 =                    Facts + dimensions + aggregated metrics
# Historical reporting	=             Timestamped fact tables + dim_time
# Machine learning	   =              Detailed fact_telemetry + historical events

# -------------------------------------------------------------------------

# This main hierarchy 

In [38]:
display(
    spark.sql("""
        CREATE OR REPLACE TABLE dbo.gold_asset_hierarchy AS

        SELECT
            a.site_id,
            b.building_id,
            a.asset_name,
            a.asset_id,
            a.asset_type

        FROM dbo.gold_dim_asset a

        INNER JOIN (
            SELECT
                building_id,
                site_id,
                ROW_NUMBER() OVER (
                    PARTITION BY site_id
                    ORDER BY building_id
                ) AS building_number
            FROM dbo.gold_dim_building
        ) b
            ON a.site_id = b.site_id

        WHERE
            (
                b.building_number = 1
                AND CAST(SUBSTRING(a.asset_id, 2) AS INT)
                    BETWEEN
                    CASE
                        WHEN a.site_id = 'S001' THEN 1
                        WHEN a.site_id = 'S002' THEN 11
                        WHEN a.site_id = 'S003' THEN 21
                        WHEN a.site_id = 'S004' THEN 31
                        WHEN a.site_id = 'S005' THEN 41
                    END
                    AND
                    CASE
                        WHEN a.site_id = 'S001' THEN 5
                        WHEN a.site_id = 'S002' THEN 15
                        WHEN a.site_id = 'S003' THEN 25
                        WHEN a.site_id = 'S004' THEN 35
                        WHEN a.site_id = 'S005' THEN 45
                    END
            )
            OR
            (
                b.building_number = 2
                AND CAST(SUBSTRING(a.asset_id, 2) AS INT)
                    BETWEEN
                    CASE
                        WHEN a.site_id = 'S001' THEN 6
                        WHEN a.site_id = 'S002' THEN 16
                        WHEN a.site_id = 'S003' THEN 26
                        WHEN a.site_id = 'S004' THEN 36
                        WHEN a.site_id = 'S005' THEN 46
                    END
                    AND
                    CASE
                        WHEN a.site_id = 'S001' THEN 10
                        WHEN a.site_id = 'S002' THEN 20
                        WHEN a.site_id = 'S003' THEN 30
                        WHEN a.site_id = 'S004' THEN 40
                        WHEN a.site_id = 'S005' THEN 50
                    END
            )
    """)
)

StatementMeta(, 8f453ac4-3c3a-42be-a14c-83dfa3de5248, 40, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 9b8d16bb-ef8a-4987-85bd-5c9e27fed33c)

In [39]:
display(spark.sql("SELECT * FROM dbo.gold_asset_hierarchy"))

StatementMeta(, 8f453ac4-3c3a-42be-a14c-83dfa3de5248, 41, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, dacfc063-e880-4150-b07b-cc58f9d8d923)

# verify  current hierarchy

In [40]:
display(
    spark.sql("""
        SELECT
            site_id,
            building_id,
            COUNT(*) AS asset_count
        FROM dbo.gold_asset_hierarchy
        GROUP BY site_id, building_id
        ORDER BY site_id, building_id
    """)
)

StatementMeta(, 8f453ac4-3c3a-42be-a14c-83dfa3de5248, 42, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 3800321e-29fe-47d3-ab1c-4bc76ccdaac4)

# check every site

In [41]:
display(
    spark.sql("""
        SELECT
            site_id,
            COUNT(DISTINCT building_id) AS building_count,
            COUNT(DISTINCT asset_id) AS asset_count
        FROM dbo.gold_asset_hierarchy
        GROUP BY site_id
        ORDER BY site_id
    """)
)

StatementMeta(, 8f453ac4-3c3a-42be-a14c-83dfa3de5248, 43, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 0af640e4-1aa1-4717-9d32-0f8dac2be895)

In [42]:
display(
    spark.sql("""
        CREATE OR REPLACE TABLE dbo.gold_asset_relationship AS

        SELECT
            site_id,
            building_id,
            parent_asset_id,
            child_asset_id,
            'PARENT_CHILD' AS relationship_type

        FROM VALUES

            -- S001 / B001
            ('S001', 'B001', 'A001', 'A002'),
            ('S001', 'B001', 'A001', 'A005'),

            -- S001 / B002
            ('S001', 'B002', 'A006', 'A007'),
            ('S001', 'B002', 'A009', 'A010'),

            -- S002 / B003
            ('S002', 'B003', 'A011', 'A012'),
            ('S002', 'B003', 'A011', 'A015'),

            -- S002 / B004
            ('S002', 'B004', 'A016', 'A017'),
            ('S002', 'B004', 'A019', 'A020'),

            -- S003 / B005
            ('S003', 'B005', 'A021', 'A022'),
            ('S003', 'B005', 'A021', 'A025'),

            -- S003 / B006
            ('S003', 'B006', 'A026', 'A027'),
            ('S003', 'B006', 'A029', 'A030'),

            -- S004 / B007
            ('S004', 'B007', 'A031', 'A032'),
            ('S004', 'B007', 'A031', 'A035'),

            -- S004 / B008
            ('S004', 'B008', 'A036', 'A037'),
            ('S004', 'B008', 'A039', 'A040'),

            -- S005 / B009
            ('S005', 'B009', 'A041', 'A042'),
            ('S005', 'B009', 'A041', 'A045'),

            -- S005 / B010
            ('S005', 'B010', 'A046', 'A047'),
            ('S005', 'B010', 'A049', 'A050')

        AS t(
            site_id,
            building_id,
            parent_asset_id,
            child_asset_id
        )
    """)
)

StatementMeta(, 8f453ac4-3c3a-42be-a14c-83dfa3de5248, 44, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 3d63c750-6827-40ea-b97a-1538c59992e0)

# Retrieve all assets under a site.

In [43]:
display(
    spark.sql("""
        SELECT
            site_id,
            building_id,
            asset_id,
            asset_name,
            asset_type

        FROM dbo.gold_asset_hierarchy

        WHERE site_id = 'S001'

        ORDER BY
            building_id,
            asset_id
    """)
)

StatementMeta(, 8f453ac4-3c3a-42be-a14c-83dfa3de5248, 45, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, a5c0e9b7-4851-44ce-bbf1-d918756bfde3)

# Retrieve Parent and Child Assets.

In [44]:
display(
    spark.sql("""
        SELECT
            r.site_id,
            r.building_id,

            r.parent_asset_id,
            p.asset_name AS parent_asset_name,
            p.asset_type AS parent_asset_type,

            r.child_asset_id,
            c.asset_name AS child_asset_name,
            c.asset_type AS child_asset_type,

            r.relationship_type

        FROM dbo.gold_asset_relationship r

        LEFT JOIN dbo.gold_dim_asset p
            ON r.parent_asset_id = p.asset_id

        LEFT JOIN dbo.gold_dim_asset c
            ON r.child_asset_id = c.asset_id

        ORDER BY
            r.site_id,
            r.building_id,
            r.parent_asset_id,
            r.child_asset_id
    """)
)

StatementMeta(, 8f453ac4-3c3a-42be-a14c-83dfa3de5248, 46, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 6f04f410-1b67-4ab7-888d-a2b215f1174a)

# Find downstream impacted assets

If Chiller-01 fails, the query identifies its downstream child assets AHU-02 and AHU-05.

In [45]:
display(
    spark.sql("""
        SELECT
            r.site_id,
            r.building_id,

            r.parent_asset_id,
            p.asset_name AS parent_asset_name,

            r.child_asset_id,
            c.asset_name AS impacted_asset_name,
            c.asset_type AS impacted_asset_type

        FROM dbo.gold_asset_relationship r

        LEFT JOIN dbo.gold_dim_asset p
            ON r.parent_asset_id = p.asset_id

        LEFT JOIN dbo.gold_dim_asset c
            ON r.child_asset_id = c.asset_id

        WHERE r.parent_asset_id = 'A001'

        ORDER BY r.child_asset_id
    """)
)

StatementMeta(, 8f453ac4-3c3a-42be-a14c-83dfa3de5248, 47, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 5be2088f-8606-4053-beb1-6ed11fa1ace3)

# Identify orphan assets

- An orphan asset is an asset that is expected to have a parent based on the hierarchy rules, but no valid parent relationship exists. For example, an AHU without its expected Chiller or Pump parent.
- All AHUs currently have a parent relationship.

Normally:

    Chiller-01
     └── AHU-02

Suppose data like that:

Building B001

 └──Chiller-01
 
└──AHU-02

Expected:

Chiller-01 (A001)

       └── AHU-02 (A002)


Actual:

Chiller-01 (A001)

AHU-02 (A002)

     Wrong
     
  No parent


just example for this:
- AHU-02 is an orphan asset because, based on our hierarchy rule, an AHU is expected to have a parent asset, but no parent relationship exists for it.

In [46]:
display(
    spark.sql("""
        SELECT
            a.site_id,
            a.asset_id,
            a.asset_name,
            a.asset_type

        FROM dbo.gold_dim_asset a

        LEFT JOIN dbo.gold_asset_relationship r
            ON a.asset_id = r.child_asset_id

        WHERE a.asset_type = 'AHU'
          AND r.child_asset_id IS NULL

        ORDER BY
            a.site_id,
            a.asset_id
    """)
)

StatementMeta(, 8f453ac4-3c3a-42be-a14c-83dfa3de5248, 48, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 3bbe1a9f-b95a-4c0b-ba0b-d8c9ffb66270)

# Identify disconnected assets

A disconnected asset is an asset that exists in the asset master but has no relationship with any other asset in the hierarchy. 

For example, A008 Pump-08 is disconnected because A008 does not appear as either a parent or a child in the asset relationship table.

Has parent OR has child

        ↓

     CONNECTED

Has NO parent AND NO child

        ↓
        
    DISCONNECTED

In [47]:
display(
    spark.sql("""
        SELECT
            a.site_id,
            a.asset_id,
            a.asset_name,
            a.asset_type

        FROM dbo.gold_dim_asset a

        LEFT JOIN dbo.gold_asset_relationship parent_rel
            ON a.asset_id = parent_rel.child_asset_id

        LEFT JOIN dbo.gold_asset_relationship child_rel
            ON a.asset_id = child_rel.parent_asset_id

        WHERE parent_rel.child_asset_id IS NULL
          AND child_rel.parent_asset_id IS NULL

        ORDER BY
            a.site_id,
            a.asset_id
    """)
)

StatementMeta(, 8f453ac4-3c3a-42be-a14c-83dfa3de5248, 49, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 8efdd213-1ecb-4a44-b534-2df0787c445a)

# **# What is partitioning?**

### **Partitioning physically organizes a large Delta table into separate folders based on a column.**

In [48]:
display(
    spark.sql("""
        DESCRIBE DETAIL dbo.gold_fact_telemetry
    """)
)

StatementMeta(, 8f453ac4-3c3a-42be-a14c-83dfa3de5248, 50, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, f68c76a9-ae9e-477d-9a6e-a31d8d8d681d)

In [49]:
display(
    spark.sql("""
        SELECT
            date_key,
            COUNT(*) AS record_count
        FROM dbo.gold_fact_telemetry
        GROUP BY date_key
        ORDER BY date_key
    """)
)

StatementMeta(, 8f453ac4-3c3a-42be-a14c-83dfa3de5248, 51, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 9bbf0ffb-97d4-4a38-b944-ad2ae46ca91b)

In [50]:
display(
    spark.sql("""
        OPTIMIZE dbo.gold_fact_telemetry
    """)
)

StatementMeta(, 8f453ac4-3c3a-42be-a14c-83dfa3de5248, 52, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 2d8fa4f6-c46f-4e3d-8f21-10e01e25df3a)

In [51]:
display(
    spark.sql("""
        OPTIMIZE dbo.gold_fact_energy
    """)
)

StatementMeta(, 8f453ac4-3c3a-42be-a14c-83dfa3de5248, 53, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 6f3407f4-548f-49f7-9ba6-be4caa3d28f2)

In [52]:
display(
    spark.sql("""
        OPTIMIZE dbo.gold_fact_event
    """)
)

StatementMeta(, 8f453ac4-3c3a-42be-a14c-83dfa3de5248, 54, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 380d8e9f-07ac-4bf7-b89f-f30babcf789a)

## Indexing strategy 

### Because our Gold layer is implemented as Delta tables in Microsoft Fabric Lakehouse, we don't use traditional clustered or non-clustered indexes like SQL Server. Instead, we optimize data access through date-based partitioning and Delta table optimization. The high-volume fact tables are partitioned by date_key, while the smaller dimension tables don't require additional indexing.

In [53]:
# CREATE INDEX IX_Asset
# ON FactTelemetry(asset_id);

StatementMeta(, 8f453ac4-3c3a-42be-a14c-83dfa3de5248, 55, Finished, Available, Finished, False)

In [54]:
# WHERE asset_id = 'A001'

StatementMeta(, 8f453ac4-3c3a-42be-a14c-83dfa3de5248, 56, Finished, Available, Finished, False)